# Loading and visualizing MuscleMimic models

This notebook introduces the basic workflow for the `musclemimic_models` package:

1. Load the two musculoskeletal models that ship with the package (`bimanual` and `myofullbody`).
2. Inspect their dimensions: joints, degrees of freedom, muscles, and tendons.
3. Render static images off-screen.
4. Render the models from several camera angles.
5. Run a short muscle-driven rollout and animate it.
6. Open the interactive viewer.

> **Rendering backend.** The first cell sets `MUJOCO_GL=egl`, which renders off-screen on a
> headless server. On a laptop or desktop with a display, remove that line before importing
> MuJoCo to use the default `glfw` backend; `mujoco.viewer.launch(...)` can then open a window.


In [ ]:
import os

# Off-screen rendering backend: use "egl" on a headless server (no display).
# On a machine with a screen, remove this line (the default "glfw" backend is used).
os.environ.setdefault("MUJOCO_GL", "egl")

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import musclemimic_models as mm

print("musclemimic_models:", mm.__version__)
print("mujoco            :", mujoco.__version__)
print("available models  :", list(mm.REGISTRY))

## 1. Loading a model

`mm.load(name)` compiles the MJCF and returns an `(MjModel, MjData)` pair, where `name` is a key in
`mm.REGISTRY`. If you prefer to load or edit the XML yourself, `mm.get_xml_path(name)` returns the
model file path.


In [ ]:
# Load by name -> (MjModel, MjData)
model, data = mm.load("bimanual")

# The underlying MJCF paths:
mm.print_path()
print("\nbimanual XML path:", mm.get_xml_path("bimanual"))

# Equivalent manual load:
#   model = mujoco.MjModel.from_xml_path(str(mm.get_xml_path("bimanual")))
#   data  = mujoco.MjData(model)

## 2. Inspecting a model

Every actuator in both models is a Hill-type **muscle** (`mjGAIN_MUSCLE`) that pulls on a
**tendon**; there are no joint-torque motors.

In [ ]:
def summarize(model):
    n_muscle = int((model.actuator_gaintype == mujoco.mjtGain.mjGAIN_MUSCLE).sum())
    return {
        "nq  (generalized coords)": model.nq,
        "nv  (degrees of freedom)": model.nv,
        "nu  (actuators)":          model.nu,
        "      of which muscles":   n_muscle,
        "ntendon":                  model.ntendon,
        "njnt (joints)":            model.njnt,
        "nbody":                    model.nbody,
    }

for name in mm.REGISTRY:
    m, _ = mm.load(name)
    print(f"=== {name} ===")
    for k, v in summarize(m).items():
        print(f"  {k:28s}: {v}")
    print()

## 3. A camera helper and a static render

`mujoco.Renderer` renders off-screen. Call `mj_forward` first so body and geom positions are current.

The `frame_camera` helper builds a free camera centered on the model's bodies. It computes a bounding
box from the model geoms and ignores the infinite floor plane; including the floor would create an
overly distant view. Adjust `azimuth` and `elevation` to view the model from different angles.


In [ ]:
def frame_camera(model, data, azimuth=-90.0, elevation=-10.0, pad=1.4):
    """A free camera framed tightly on the model's bodies (ignores the floor)."""
    mujoco.mj_forward(model, data)
    geom_ids = [i for i in range(model.ngeom)
                if mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, i) != "floor"]
    pts = data.geom_xpos[geom_ids]
    lo, hi = pts.min(0), pts.max(0)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = 0.5 * (lo + hi)
    cam.distance = pad * float(np.linalg.norm(hi - lo))
    cam.azimuth, cam.elevation = azimuth, elevation
    return cam


def make_renderer(model, width, height):
    """Renderer with the off-screen framebuffer enlarged to fit (default is 640x480)."""
    model.vis.global_.offwidth = max(int(model.vis.global_.offwidth), width)
    model.vis.global_.offheight = max(int(model.vis.global_.offheight), height)
    return mujoco.Renderer(model, height, width)


def render(model, data, camera=None, width=640, height=480):
    mujoco.mj_forward(model, data)
    if camera is None:
        camera = frame_camera(model, data)
    with make_renderer(model, width, height) as r:
        r.update_scene(data, camera=camera)
        return r.render()


model, data = mm.load("bimanual")
img = render(model, data, frame_camera(model, data, azimuth=-90, pad=1.25))
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis("off")
plt.title("BimanualMuscle - default pose")
plt.show()

## 4. The full-body model

`myofullbody` is free-floating (it has a free joint at the root) and ships with a keyframe for a
natural standing pose. Reset to it before rendering.

In [ ]:
model, data = mm.load("myofullbody")
if model.nkey > 0:
    mujoco.mj_resetDataKeyframe(model, data, 0)   # natural standing pose

img = render(model, data, frame_camera(model, data, azimuth=120, elevation=-10, pad=1.15),
             width=480, height=640)
plt.figure(figsize=(5, 7))
plt.imshow(img)
plt.axis("off")
plt.title("MyoFullBody - keyframe pose")
plt.show()

## 5. Multiple camera angles

In [ ]:
model, data = mm.load("bimanual")
mujoco.mj_forward(model, data)

azimuths = [-130, -60, 0, 90]
fig, axes = plt.subplots(1, len(azimuths), figsize=(4 * len(azimuths), 4))
for ax, az in zip(axes, azimuths):
    ax.imshow(render(model, data, frame_camera(model, data, azimuth=az, pad=1.25)))
    ax.set_title(f"azimuth = {az}\u00b0")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. A muscle-driven rollout

For this model, `data.ctrl` is the muscle control input. The bimanual XML allows controls in
`[-1, 1]`; this example uses a smooth nonnegative excitation in `[0, 1]` to drive the right-elbow
flexors (biceps, brachialis, and brachioradialis), then collects rendered frames.

`mujoco.mj_step` advances the simulation by `model.opt.timestep` seconds.


In [ ]:
model, data = mm.load("bimanual")
mujoco.mj_resetData(model, data)

# Right-side elbow flexors (no name suffix == right arm).
flexors = ["BIClong", "BICshort", "BRA", "BRD"]
flexor_ids = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in flexors]
missing = [name for name, idx in zip(flexors, flexor_ids) if idx < 0]
if missing:
    raise ValueError(f"Missing actuators: {missing}")
elbow_qadr = model.jnt_qposadr[
    mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, "elbow_flex_r")]

cam = frame_camera(model, data, azimuth=-90, pad=1.2)
fps, duration = 30, 3.0
frames, elbow_deg = [], []

with make_renderer(model, 480, 360) as r:
    while data.time < duration:
        a = 0.5 * (1 - np.cos(2 * np.pi * 0.5 * data.time))   # smooth 0 -> 1 -> 0
        data.ctrl[:] = 0.0
        data.ctrl[flexor_ids] = a
        mujoco.mj_step(model, data)

        if len(frames) < data.time * fps:                      # subsample to `fps`
            r.update_scene(data, camera=cam)
            frames.append(r.render())
            elbow_deg.append(np.rad2deg(data.qpos[elbow_qadr]))

print(f"{len(frames)} frames; right elbow flexion "
      f"{min(elbow_deg):.0f}\u00b0 -> {max(elbow_deg):.0f}\u00b0")

In [ ]:
# A strip of frames through the rollout.
idx = np.linspace(0, len(frames) - 1, 6).astype(int)
fig, axes = plt.subplots(1, len(idx), figsize=(3 * len(idx), 3))
for ax, i in zip(axes, idx):
    ax.imshow(frames[i])
    ax.set_title(f"t = {i / fps:.2f}s")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Inline animation (optional)

`to_jshtml()` embeds a small JavaScript player, so it does not require ffmpeg. This cell needs
IPython/Jupyter.


In [ ]:
from matplotlib import animation

fig = plt.figure(figsize=(5, 4))
plt.axis("off")
im = plt.imshow(frames[0])
anim = animation.FuncAnimation(
    fig, lambda f: (im.set_data(f), (im,))[1], frames=frames,
    interval=1000 / fps, blit=True,
)
plt.close(fig)

from IPython.display import HTML
HTML(anim.to_jshtml())

## 7. Interactive viewer

On a machine with a display, the viewer opens a window that you can orbit and zoom with the mouse.
This will not work on a headless server.


In [ ]:
import mujoco.viewer

model, data = mm.load("bimanual")

# Blocks until you close the window -- run this OUTSIDE a headless server:
#   mujoco.viewer.launch(model, data)
#
# Or drive a passive viewer that you step yourself:
#   with mujoco.viewer.launch_passive(model, data) as viewer:
#       while viewer.is_running():
#           mujoco.mj_step(model, data)
#           viewer.sync()
print("Uncomment the lines above and run on a machine with a display.")